# Atividade: A2C (Advantage Actor Critic) com um braço robótico 🦾 usando Panda-Gym

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/thumbnail.jpg" alt="Unit 6 Thumbnail">

Neste notebook, **você treinará um agente Advantage Actor Critic (A2C)** para controlar um braço robótico Panda 🐼 na simulação [Panda-Gym](https://github.com/qgallouedec/panda-gym), usando o [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/).

Este notebook é uma simplificação do unit6.ipynb do **Curso de Aprendizagem por Reforço Profundo** (Deep Reinforcement Learning) https://huggingface.co/datasets/huggingface-deep-rl-course/.

> ⚠️ **Versão adaptada para as versões atuais** (gymnasium ≥ 1.0, stable-baselines3 ≥ 2.0, panda-gym 3.x com MuJoCo). O treino usa a GPU do Colab.

⬇️ Aqui está um exemplo do que **você vai alcançar** ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/a2c-PandaReachDense-v3/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 Ambiente:

- [Panda-Gym](https://github.com/qgallouedec/panda-gym) — simulação realista de um braço robótico Franka Emika Panda (MuJoCo).

### 📚 Biblioteca de RL:

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) — implementação do A2C que usamos.

## Objetivos deste notebook 🏆
No final do notebook, você irá:
- Aprender a usar o **Panda-Gym** e entender o espaço de observação em **dicionário** (Dict).
- Treinar um agente **A2C** com o Stable-Baselines3.
- **Normalizar** observações e recompensas (`VecNormalize`).
- **Avaliar o agente e gravar um vídeo** do braço robótico completando a tarefa.

## Instalar as dependências 🔽

Precisamos do **stable-baselines3**, do **panda_gym** (que traz o simulador MuJoCo junto) e do **imageio-ffmpeg** (para gravar o vídeo). O gymnasium e o PyTorch já vêm no Colab.

In [ ]:
%pip install -q stable-baselines3 panda_gym moviepy

# - stable-baselines3: o algoritmo A2C + utilitários (vec envs, normalização)
# - panda_gym: simulação do braço robótico (instala o MuJoCo junto)
# - moviepy: gravação do vídeo de replay (VecVideoRecorder)

## Importar os pacotes 📦

In [ ]:
import os

import gymnasium as gym
import panda_gym  # registra os ambientes PandaReachDense-v3 etc. no gymnasium

# Stable-Baselines3
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.evaluation import evaluate_policy

# PandaReachDense-v3 🦾

## Criar o ambiente 🎮

O agente é um **braço robótico** que precisa mover a garra até uma posição-alvo:

- **Observação**: um dicionário com 3 elementos (`achieved_goal`: posição atual da garra; `desired_goal`: posição alvo; `observation`: dados internos do braço).
- **Ação**: vetor de 3 valores — o deslocamento da garra nos eixos x, y e z.
- **Recompensa** (versão *dense*): negativa e proporcional à distância até o alvo — quanto mais perto, menos negativa (0 é o melhor).

In [ ]:
env_id = "PandaReachDense-v3"

# Cria o ambiente
env = gym.make(env_id)

# Espaços de observação e de ação
s_size = env.observation_space
a_size = env.action_space

In [ ]:
print("_____ESPAÇO DE OBSERVAÇÃO_____\n")
print("Tipo:", type(s_size))
print("Chaves do dicionário:", s_size.spaces.keys())
print("Exemplo de observação:")
print(env.observation_space.sample())

print("\n_____ESPAÇO DE AÇÃO_____\n")
print("Ação:", a_size)
print("Exemplo de ação:", env.action_space.sample())

A observação é um **dicionário com 3 chaves**: `achieved_goal` (posição atual da garra, x,y,z), `desired_goal` (posição alvo, x,y,z) e `observation` (estado interno do braço). Por isso o modelo usará a política `MultiInputPolicy`, que aceita observações em dicionário.

## Normalizar observações e recompensas 📏

Uma boa prática em RL é **normalizar** as entradas da rede: a observação tem grandezas muito diferentes (posições em metros, velocidades em m/s…), e a recompensa muda de escala ao longo do treino. O wrapper `VecNormalize` mantém média/desvio-padrão dos dados e normaliza tudo.

Também **vetorizamos** o ambiente: 4 cópias em paralelo deixam o treino mais rápido e estável.

✏️ **Exercício**: complete a linha do wrapper de normalização.

In [ ]:
# Exercício: complete os espaços e descomente.
# env = make_vec_env(env_id, n_envs=4)
# env = ???(env, norm_obs=True, norm_reward=True, clip_obs=10.)

<details><summary><b>Solução</b></summary>

```python
env = make_vec_env(env_id, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)
```
</details>

In [ ]:
# Solução
env = make_vec_env(env_id, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

## Criar o modelo A2C 🤖

O A2C (Advantage Actor Critic) combina:
- um **ator** (política) que escolhe as ações;
- um **crítico** (função valor) que estima o valor dos estados;
- a **vantagem** `A(s,a) = Q(s,a) − V(s)` diz o quanto cada ação é melhor que a média — e é usada para atualizar a política.

✏️ **Exercício**: crie o modelo. (Dica: a observação é um dicionário, então a política é `MultiInputPolicy`.)

In [ ]:
# Exercício: complete os espaços e descomente.
# model = A2C(policy="???", env=env, verbose=1)

<details><summary><b>Solução</b></summary>

```python
model = A2C(policy="MultiInputPolicy", env=env, verbose=1)
```
</details>

In [ ]:
# Solução
model = A2C(policy="MultiInputPolicy", env=env, verbose=1)

## Treinar o agente 🏃

Treinamos por **1.000.000 de timesteps**. Com a GPU do Colab, o treino é tranquilo — aproveite e acompanhe os logs no output da célula.

In [ ]:
model.learn(1_000_000)

## Salvar o modelo 💾

Salvamos o modelo **e as estatísticas de normalização** — sem elas, o modelo não funciona depois.

In [ ]:
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")
print("Modelo e estatísticas salvos ✓")

## Avaliar o agente 📈

Para avaliar corretamente:
- carregamos o modelo e as estatísticas do `VecNormalize`;
- criamos um ambiente de avaliação com `render_mode="rgb_array"` (para poder gravar);
- `training=False` e `norm_reward=False` — na avaliação as estatísticas não são atualizadas e a recompensa volta à escala original.

In [ ]:
# Carrega as estatísticas salvas
eval_env = DummyVecEnv([lambda: gym.make(env_id)])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# Precisamos sobrescrever o render_mode para poder gravar o vídeo
eval_env.render_mode = "rgb_array"

# Não atualiza as estatísticas na avaliação
eval_env.training = False
# Não normaliza a recompensa na avaliação
eval_env.norm_reward = False

# Carrega o agente treinado
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Recompensa média = {mean_reward:.2f} +/- {std_reward:.2f}")

## Gravar um vídeo do replay 🎥

A função abaixo grava o agente jogando uma partida usando o `VecVideoRecorder` do Stable-Baselines3.

In [ ]:
from stable_baselines3.common.vec_env import VecVideoRecorder

def record_video(eval_env, model, video_folder="videos-a2c", video_length=500):
    """
    Grava um vídeo de replay do agente treinado
    :param eval_env: ambiente de avaliação (já com VecNormalize e render_mode="rgb_array")
    :param model: modelo A2C treinado
    :param video_folder: pasta de saída
    :param video_length: número de passos a gravar
    """
    # Grava os primeiros video_length passos
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda x: x == 0,
        video_length=video_length,
        name_prefix="a2c-panda",
    )

    obs = eval_env.reset()
    for _ in range(video_length + 1):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, _, _ = eval_env.step(action)

    eval_env.close()
    print(f"Vídeo salvo em {video_folder}/ (arquivo .mp4)")

In [ ]:
record_video(eval_env, model)

## Alguns desafios adicionais 🏆

A melhor maneira de aprender **é tentar as coisas por conta própria**!

Por que não tentar o **`PandaPickAndPlace-v3`** — a tarefa em que o braço precisa **pegar um cubo e colocá-lo no alvo**? Os passos são os mesmos: criar o ambiente vetorizado, normalizar, criar o A2C, treinar, salvar e avaliar.

*(Em robótica real, usam-se algoritmos mais eficientes em amostras, como SAC ou TQC — mover demais um braço de verdade tem risco de quebrá-lo! Na simulação, o A2C dá conta.)*

<details><summary><b>Solução do desafio</b></summary>

```python
env_id = "PandaPickAndPlace-v3"
env = make_vec_env(env_id, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

model = A2C(policy="MultiInputPolicy", env=env, verbose=1)
model.learn(1_000_000)

model.save("a2c-PandaPickAndPlace-v3")
env.save("vec_normalize.pkl")

eval_env = DummyVecEnv([lambda: gym.make(env_id)])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)
eval_env.render_mode = "rgb_array"
eval_env.training = False
eval_env.norm_reward = False

model = A2C.load("a2c-PandaPickAndPlace-v3")
mean_reward, std_reward = evaluate_policy(model, eval_env)
print(f"Recompensa média = {mean_reward:.2f} +/- {std_reward:.2f}")

record_video(eval_env, model)
```
</details>

________________________________________________________________________
Parabéns por terminar esta atividade! 🥳

### Continue Aprendendo, Continue Incrível 🤗